In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [ ]:
data = pd.read_csv("/home/aj/GenAI/dl_for_nlp/Data/Churn_Modelling.csv")
data.head()

In [ ]:
# drop irrelevant features
# "RowNumber", "CustomerId", "Surname", which is irrelevant to the business logic
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
data.head()

In [ ]:
# LABEL ENCODERS
# now look at columns that can be converted into categories  
# say "Geography", "Gender"

# GENDER ENCODING
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
data

In [ ]:
# GEOGRAPHICAL DATA ENCODING
from sklearn.preprocessing import OneHotEncoder
label_encoder_geo_data = OneHotEncoder()
geo_encoded_data = label_encoder_geo_data.fit_transform(data[['Geography']])
geo_encoded_data.toarray()

In [ ]:
geo_column_names = label_encoder_geo_data.get_feature_names_out()
geo_column_names

In [ ]:
# make a data frame from identified features and the encoded matrices
geo_encoded_df = pd.DataFrame(geo_encoded_data.toarray(), columns=geo_column_names)
geo_encoded_df

In [ ]:
# after encoding geographhcal data add back to orignal data also 
data = pd.concat([data, geo_encoded_df], axis=1)
data

In [ ]:
# deop the orignal column ie.'Geography'
data = data.drop('Geography',axis=1)
data

In [ ]:
data.to_pickle("/home/aj/GenAI/dl_for_nlp/Data/Churn_Modelling.pkl")

In [9]:
pkl_file = "/home/aj/GenAI/dl_for_nlp/Data/Churn_Modelling.pkl"
data = pd.read_pickle(pkl_file)
# data

In [10]:
# select your dependent and indepenedt features
# in our case 'Exited' is dependent frature ==> Y axis
# and rest all colums are independent features ==> x axis
x = data.drop('Exited', axis=1)
y = data['Exited']


In [11]:
# split data in training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2, random_state=42)

In [12]:
# scale the features
# WE WILL SCALE ONLY THE TRAIN DATA NOT THE TEST DATA 
# TO AVOID THE INFLUENCE OF TEST DATA ON TRAINING (DATA LEAKAGE). We must NEVER learn anything from test data.
scaler = StandardScaler()
scaler.fit(x_train)
x_train = scaler.transform(x_train)
# x_train

In [13]:
x_test = scaler.transform(x_test)
# x_test

In [14]:
# CHECKPOINT 
with open ("/home/aj/GenAI/dl_for_nlp/Data/Churn_Modelling_scaler.pkl", 'wb')as file:
    pickle.dump(scaler, file)

# Learning Network Making

## 1) Define the model

In [15]:
# layers
# 1.input layer
# 2.hiddenlayer 1 + activation  -->primary layer connected with input layer
# 3.hidden layer 2 + activation -->
# .
# .
# .
# n.output layer --> choose activation function conciously

In [16]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [36]:
input_shape = x_train.shape[1]
input_shape

12

In [18]:
ann_model_classic = Sequential([
    Dense(units=64, activation='relu', input_shape=(input_shape,)),
    Dense(units=32, activation='relu'),
    Dense(units=1, activation='sigmoid')
])

/home/aj/miniconda3/envs/genai/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-11-20 15:39:11.823270: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_UNKNOWN: unknown error
2025-11-20 15:39:11.823423: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2025-11-20 15:39:11.823441: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: aj-device
2025-11-20 15:39:11.823453: I external/local

In [37]:
ann_model_classic.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
ann_model_modern = Sequential([
    tf.keras.Input(shape=(input_shape,)),
    Dense(units=64, activation='relu'),
    Dense(units=32, activation='relu'),
    Dense(units=1, activation='sigmoid')
])

In [ ]:
ann_model_modern.summary()

## 2) compile the model

In [22]:
# 1)make the optimizer (def)
adam_optmizer = tf.keras.optimizers.Adam(learning_rate=0.01)

# 2)decide loss function
cross_entropy_loss = tf.keras.losses.BinaryCrossentropy()

# 3)compile the model
ann_model_modern.compile(optimizer=adam_optmizer, loss=cross_entropy_loss, metrics=['accuracy'])

In [23]:
# 4)setup the tensor board --> metrices you want to see
logs = "/home/aj/GenAI/dl_for_nlp/Data/logs/ann_" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
Tensor_board_callback = TensorBoard(log_dir=logs, histogram_freq=1)

In [27]:
# 5) Set up early stopping --> we can train our model for any number of epochs 
# There must be a method by which we can stop our model before overfitting
# For this, we have to continuously monitor our loss value and stop our model at the best accuracy point
# or say after a certain epoch loss is not decreasing, we should stop the model training
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [39]:
# 5) train the model finally
history = ann_model_modern.fit(
    x=x_train,
    y=y_train, 
    validation_data=(x_test,y_test), 
    epochs=100, 
    callbacks=[Tensor_board_callback,early_stopping_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8708 - loss: 0.3183 - val_accuracy: 0.8585 - val_loss: 0.3501
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8655 - loss: 0.3176 - val_accuracy: 0.8655 - val_loss: 0.3501
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8689 - loss: 0.3142 - val_accuracy: 0.8615 - val_loss: 0.3606
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8727 - loss: 0.3114 - val_accuracy: 0.8630 - val_loss: 0.3439
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8721 - loss: 0.3124 - val_accuracy: 0.8540 - val_loss: 0.3801
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8734 - loss: 0.3114 - val_accuracy: 0.8610 - val_loss: 0.3646
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8742 - loss: 0.3069 - val_accuracy: 0.8650 - val_loss: 0.3608
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8733 - loss: 0.3039 - val_accu

In [30]:
# Legacy format
'''
HDF5 format limitations:
    1)Cannot always store custom layers properly
    2)Harder to make backward-compatible
    3)Slower for large models
    4)Not fully compatible with TF’s SavedModel internals
'''
ann_model_modern.save(filepath='/home/aj/GenAI/dl_for_nlp/Data/models/churn_ann_model.h5')

In [31]:
# Advanced format
'''
New 'Keras' format advantages
    1)preserves model architecture
    2)preserves weights
    3)preserves training configuration
    4)preserves optimizer state
    5)supports newer Keras features
    6)It is more stable for future versions
'''
ann_model_modern.save(filepath='/home/aj/GenAI/dl_for_nlp/Data/models/churn_ann_model.keras')

In [ ]:
'''
Why .keras is better than .h5
Feature                      .h5 (legacy)           .keras (new)
Architecture	             Stored	                Stored
Weights	                     Stored             	Stored
Optimizer state	             partial/broken     	full
Custom objects	             problematic        	clean
Human-readable	             No	                    JSON-based
Cross-version stable    	 No                 	Yes
Single-file zip	             YES                    YES
Officially recommended	     NO	                    YES
'''

# internal structure of Keras file format
'''
1)full format:
    my_model.keras/
    │
    ├── metadata.json
    ├── model.json
    ├── assets/
    │   └── tokenizer.json
    └── variables/
        ├── variables.data-00000-of-00001
        └── variables.index
2) minimal format:
    my_model.keras/
    │
    ├── config.json
    ├── metadata.json
    └── weights/
        ├── dense.kernel.npy
        ├── dense.bias.npy
        ├── conv.kernel.npy
        └── conv.bias.npy
'''